# Transformação de dados , limpeza de dados  nulos ou duplicatas (PySpark)

Lê o JSON bruto salvo em `data/raw/` feito na `ingestion/extract_ibge.py`,
aplica limpeza (remoção da linha de cabeçalho da API,
 tipagem, tratamento de nulos/duplicatas) e 
grava o resultado em `data/processed/` em
formato Parquet.


In [ ]:
# Importa a classe SparkSession, responsável por criar e gerenciar
# a sessão do Apache Spark. É o ponto de entrada para utilizar
# todas as funcionalidades do PySpark.
from pyspark.sql import SparkSession

# Importa o módulo de funções do PySpark e cria um apelido (F).
# Esse módulo fornece funções para transformar dados, como:
# col(), when(), sum(), avg(), upper(), lower(), etc.
import pyspark.sql.functions as F

# Importa a biblioteca JSON do Python.
# Será utilizada para ler arquivos JSON e converter objetos Python
# para o formato JSON quando necessário.
import json

# Importa a biblioteca responsável por manipular arquivos,
# diretórios e caminhos do sistema operacional.
import os

# Cria uma SparkSession.
# builder -> inicia a configuração da sessão.
# appName() -> define o nome da aplicação que aparecerá na Spark UI.
# getOrCreate() -> cria uma nova sessão caso ela não exista;
# caso já exista uma sessão ativa, reutiliza a mesma.
spark = (
    SparkSession.builder
    .appName("transformacao-ibge")
    .getOrCreate()
)

# Define o caminho da pasta onde estão armazenados
# os arquivos JSON brutos extraídos da API do IBGE.
RAW_DIR = "/home/jovyan/work/data/raw"

# Define o caminho da pasta onde serão salvos
# os dados já transformados e prontos para as próximas
# etapas do pipeline ETL.
PROCESSED_DIR = "/home/jovyan/work/data/processed"

##  Descobrir o arquivo bruto mais recente

Em vez de fixar o nome do arquivo (que muda a cada ingestão, por
causa do timestamp), lemos o `_metadata.json` ,assim o notebook sempre processa a
ingestão mais recente.

In [ ]:
with open(os.path.join(RAW_DIR, "_metadata.json"), "r", encoding="utf-8") as f:
    historico = json.load(f)

ultima_ingestao = historico[-1]
caminho_raw = os.path.join(RAW_DIR, ultima_ingestao["arquivo"])

print("Arquivo bruto a processar:", caminho_raw)
print("Registros esperados:", ultima_ingestao["quantidade_registros"])

##  Ler o JSON bruto com Spark

In [ ]:
ddf_raw = spark.read.option("multiline", "true").json(caminho_raw)

print("Total de linhas lidas (inclui a linha de cabeçalho da API):", ddf_raw.count())
ddf_raw.printSchema()
ddf_raw.show(5, truncate=40)

## Remover a linha de cabeçalho da API

A linha de cabeçalho tem `D1C` como texto descritivo
(`"Município (Código)"`), enquanto as linhas de dado real têm `D1C`
numérico (o código IBGE do município). Filtramos usando essa
diferença.

In [ ]:
df_dados = ddf_raw.filter(F.col("D1C").rlike("^[0-9]+$"))

linhas_removidas = ddf_raw.count() - df_dados.count()
print(f"Linhas de cabeçalho removidas: {linhas_removidas}")
print(f"Linhas de dado real: {df_dados.count()}")

##  Selecionar, renomear e tipar as colunas

A API devolve tudo como string. Aqui convertemos para os tipos
corretos e damos nomes legíveis às colunas.

In [ ]:
df_limpo = (
    df_dados
    .select(
        F.col("D1C").cast("int").alias("municipio_codigo"),
        F.col("D1N").alias("municipio_nome"),
        F.col("D2C").cast("int").alias("ano"),
        F.col("MN").alias("unidade_medida"),
        # valores como '...', '-' ou 'X' viram nulo; o resto vira numero
        F.when(F.col("V").rlike("^[0-9]+$"), F.col("V").cast("long"))
         .otherwise(F.lit(None))
         .alias("populacao_estimada"),
    )
)

df_limpo.show(5)
df_limpo.printSchema()